In [1]:
# Gaps along the major axis: how many galaxies have gas beyond a gap, and how much.
# Same map and row as the walk in barolo.major_axis_extent: the DATA moment-1 map of the 3-ring
# BBarolo fit, row through the cube centre (PA = 90 deg, so the major axis is horizontal).
import numpy as np
import pandas as pd
from pathlib import Path
from astropy.io import fits

MARTINI = Path("/scratch/s4636708/aida/derived/martini")
SNAP_Z = {17: 5, 21: 4, 25: 3, 33: 2, 50: 1, 67: 0.5}
BEAM_PX = 6 # 30 arcsec beam / 5 arcsec pixels: gaps narrower than this are stepped over
PX_ARCSEC = 5

In [2]:
def major_axis(model, z, sub):
    """mom1 and mom0 along the major axis: (left side, right side), each from the centre outward."""
    maps = MARTINI / f"z{z:g}" / model / f"gal_{sub:06d}" / "bbarolo_3rings" / "maps"
    mom1 = np.squeeze(fits.getdata(maps / "MOCK_1mom.fits"))
    mom0 = np.squeeze(fits.getdata(maps / "MOCK_0mom.fits"))
    ny, nx = mom1.shape
    x0, y0 = int((nx - 1) / 2), int((ny - 1) / 2) # centre pixel, as in the walk
    left = (mom1[y0, x0::-1], mom0[y0, x0::-1])
    right = (mom1[y0, x0:], mom0[y0, x0:])
    return left, right


def walk(mom1, mom0, step_px):
    """Walk out from the centre along one side, skipping the blank centre (hole), and stop at the first
    gap of step_px blank pixels or more (step_px = 1: stop at the first blank pixel, the current rule).
    A pixel has emission if its mom1 is not NaN, i.e. it is inside BBarolo's mask.
    Returns where the walk stopped, whether there is emission after that gap, and the share of this
    side's mom0 flux that lies beyond the stop."""
    has = np.isfinite(mom1)
    flux = np.nan_to_num(mom0)
    if not has.any(): # no emission on this side
        return dict(stop_arcsec=0.0, gas_beyond=False, frac_beyond=0.0)
    i = int(np.argmax(has)) # first pixel with emission: end of the hole
    while True:
        blank = np.flatnonzero(~has[i:])
        if len(blank) == 0: # emission up to the cube edge
            return dict(stop_arcsec=float((len(has) - 1) * PX_ARCSEC), gas_beyond=False, frac_beyond=0.0)
        g0 = i + int(blank[0]) # first blank pixel: start of a gap
        after = np.flatnonzero(has[g0:])
        if len(after) == 0: # nothing after the gap: the gas just ends here
            return dict(stop_arcsec=float((g0 - 1) * PX_ARCSEC), gas_beyond=False, frac_beyond=0.0)
        width = int(after[0]) # gap width in pixels
        if width < step_px: # narrow gap: step over it and keep walking
            i = g0 + width
            continue
        beyond = flux[g0:][has[g0:]].sum() / flux[has].sum()
        return dict(stop_arcsec=float((g0 - 1) * PX_ARCSEC), gas_beyond=True, frac_beyond=float(beyond))

In [ ]:
# every galaxy in the manifest (~8 min, reading the maps from /scratch): both sides with the current rule
# (stop at the first blank pixel) and with the new one (step over gaps narrower than a beam)
rows = []
for line in open(MARTINI / "manifest.txt"):
    model, snap, sub = line.split()[:3]
    z, sub = SNAP_Z[int(snap)], int(sub)
    try:
        sides = major_axis(model, z, sub)
    except OSError:
        print("no moment maps:", model, z, sub)
        continue
    now = [walk(m1, m0, 1) for m1, m0 in sides]
    new = [walk(m1, m0, BEAM_PX) for m1, m0 in sides]
    rows.append(dict(model=model, z=z, subID=sub,
                     cut_short_now=now[0]["gas_beyond"] or now[1]["gas_beyond"],  # current walk stops before some gas
                     gas_beyond=new[0]["gas_beyond"] or new[1]["gas_beyond"],     # gas beyond a gap >= 1 beam
                     frac_beyond=max(new[0]["frac_beyond"], new[1]["frac_beyond"]), # larger of the two sides
                     extent_now=[now[0]["stop_arcsec"], now[1]["stop_arcsec"]],
                     extent_new=[new[0]["stop_arcsec"], new[1]["stop_arcsec"]]))
gaps = pd.DataFrame(rows)
gaps

In [4]:
# counts per redshift
#   cut_short_now: the current walk stops before some of the gas
#   fixed_by_step: cut short now, but no gas left beyond a gap >= 1 beam once the narrow gaps are stepped over
#   gas_beyond:    still gas beyond a gap >= 1 beam
#   beyond_XXpc:   of those, more than XX% of the side's flux beyond the gap
def counts(g):
    return pd.Series(dict(N=len(g),
                          cut_short_now=g["cut_short_now"].sum(),
                          fixed_by_step=(g["cut_short_now"] & ~g["gas_beyond"]).sum(),
                          gas_beyond=g["gas_beyond"].sum(),
                          beyond_10pc=(g["frac_beyond"] > 0.10).sum(),
                          beyond_25pc=(g["frac_beyond"] > 0.25).sum(),
                          beyond_50pc=(g["frac_beyond"] > 0.50).sum()))

cols = ["cut_short_now", "gas_beyond", "frac_beyond"]
by_z = gaps.groupby("z")[cols].apply(counts)
by_z.loc["all"] = counts(gaps)
by_z

,N,cut_short_now,fixed_by_step,gas_beyond,beyond_10pc,beyond_25pc,beyond_50pc
z,,,,,,,
0.5,1424,561,102,459,178,130,99
1.0,1339,389,87,302,92,58,42
2.0,854,165,57,108,18,15,13
3.0,376,72,24,48,6,4,3
4.0,80,11,7,4,0,0,0
5.0,8,0,0,0,0,0,0
all,4081,1198,277,921,294,207,157


In [5]:
# the same per redshift and model
gaps.groupby(["z", "model"])[cols].apply(counts)

N  cut_short_now  fixed_by_step  gas_beyond  beyond_10pc  \
z   model                                                               
0.5 CDM    460            184             34         150           63   
    SIDM1  503            189             35         154           60   
    vSIDM  461            188             33         155           55   
1.0 CDM    425            126             31          95           31   
    SIDM1  477            132             28         104           30   
    vSIDM  437            131             28         103           31   
2.0 CDM    259             52             22          30            6   
    SIDM1  312             48             16          32            5   
    vSIDM  283             65             19          46            7   
3.0 CDM    122             27             13          14            1   
    SIDM1  138             22              5          17            2   
    vSIDM  116             23              6          17            3   
4.0 CDM     23              5              3           2            0   
    SIDM1   33              3              2           1            0   
    vSIDM   24              3              2           1            0   
5.0 CDM      1              0              0           0            0   
    SIDM1    4              0              0           0            0   
    vSIDM    3              0              0           0            0   

           beyond_25pc  beyond_50pc  
z   model                            
0.5 CDM             49           39  
    SIDM1           43           30  
    vSIDM           38           30  
1.0 CDM             18           14  
    SIDM1           24           17  
    vSIDM           16           11  
2.0 CDM              5            5  
    SIDM1            3            2  
    vSIDM            7            6  
3.0 CDM              1            0  
    SIDM1            1            1  
    vSIDM            2            2  
4.0 CDM              0            0  
    SIDM1            0            0  
    vSIDM            0            0  
5.0 CDM              0            0  
    SIDM1            0            0  
    vSIDM            0            0

In [6]:
# one galaxy: '#' = pixel with emission, '.' = blank, from the centre outward (1 character = 5 arcsec)
model, z, sub = "CDM", 0.5, 91726
for name, (m1, m0) in zip(("left", "right"), major_axis(model, z, sub)):
    print(name.ljust(6) + "".join("#" if np.isfinite(v) else "." for v in m1))
    print(" " * 6, walk(m1, m0, BEAM_PX))

left  #################.....................##################################################################################################################################
       {'stop_arcsec': 80.0, 'gas_beyond': True, 'frac_beyond': 0.399118036031723}
right ##############.......................#################.....................................##############################################################################
       {'stop_arcsec': 65.0, 'gas_beyond': True, 'frac_beyond': 0.5134053230285645}
